# HGP-Clusterer 3D : démonstration

Ce notebook applique `HGPClusterer` à un nuage 3D jouet : deux spirales entrelacées, un amas gaussien et un bruit uniforme. Il compare le résultat à HDBSCAN et montre comment raffiner les clusters sans recalculer la triangulation.

Documentation : `README.md`, `docs/ALGORITHME.md` et `docs/API.md` du dossier `HGP-Clusterer3D`.

## Installation

Dans Google Colab, déposer d'abord le dossier `HGP-Clusterer3D` (ou son archive `HGP-Clusterer3D.zip`) dans `/content`. La cellule suivante installe alors Eigen et TBB, compile Geogram 1.10.1 et installe le paquet (quelques minutes). Hors de Colab, elle suppose le paquet déjà installé (voir le README), ainsi que Jupyter, `matplotlib` et `scikit-learn>=1.3`.

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
GEOGRAM_VERSION = "v1.10.1"
PACKAGE_DIR = "/content/HGP-Clusterer3D"


def run(command):
    print("$", command)
    subprocess.run(command, shell=True, check=True)


if IN_COLAB:
    run("apt-get -qq update && apt-get -qq install -y build-essential cmake git unzip libeigen3-dev libtbb-dev")
    if not os.path.exists("/usr/local/include/geogram1"):
        run(f"git clone --depth 1 --branch {GEOGRAM_VERSION} --recurse-submodules https://github.com/BrunoLevy/geogram.git /content/geogram")
        run(
            "cmake -S /content/geogram -B /content/geogram/build -DCMAKE_BUILD_TYPE=Release -DGEOGRAM_LIB_ONLY=ON "
            "-DGEOGRAM_WITH_GRAPHICS=OFF -DGEOGRAM_WITH_LUA=OFF -DGEOGRAM_WITH_LEGACY_NUMERICS=OFF "
            "-DGEOGRAM_WITH_HLBFGS=OFF -DGEOGRAM_WITH_TETGEN=OFF -DGEOGRAM_WITH_TRIANGLE=OFF"
        )
        run("cmake --build /content/geogram/build --parallel 4")
        run("cmake --install /content/geogram/build --prefix /usr/local && ldconfig")
    if not os.path.isdir(PACKAGE_DIR) and os.path.isfile(PACKAGE_DIR + ".zip"):
        run(f"unzip -q {PACKAGE_DIR}.zip -d /content")
    if not os.path.isdir(PACKAGE_DIR):
        raise FileNotFoundError(f"déposer le dossier HGP-Clusterer3D (ou HGP-Clusterer3D.zip) dans /content")
    run(f"pip install -q {PACKAGE_DIR}")

## Nuage jouet

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import HDBSCAN
from sklearn.metrics import adjusted_rand_score

from hgp_clusterer import HGPClusterer

rng = np.random.default_rng(0)
n = 1500
t = rng.uniform(0, 4 * np.pi, n)
spiral_a = np.c_[np.cos(t), np.sin(t), t / (2 * np.pi)] + rng.normal(scale=0.05, size=(n, 3))
t = rng.uniform(0, 4 * np.pi, n)
spiral_b = np.c_[np.cos(t + np.pi), np.sin(t + np.pi), t / (2 * np.pi)] + rng.normal(scale=0.05, size=(n, 3))
blob = rng.normal(loc=(3.0, 0.0, 1.0), scale=0.25, size=(800, 3))
noise = rng.uniform(low=(-2.0, -2.0, -0.5), high=(4.0, 2.0, 2.5), size=(600, 3))

X = np.vstack([spiral_a, spiral_b, blob, noise])
y = np.repeat([0, 1, 2, -1], [n, n, 800, 600])
core = y >= 0
print(X.shape)

In [ ]:
def show(panels, title=None):
    fig = plt.figure(figsize=(5 * len(panels), 4.5))
    for i, (name, labels) in enumerate(panels.items(), start=1):
        ax = fig.add_subplot(1, len(panels), i, projection="3d")
        noise = labels < 0
        ax.scatter(*X[noise].T, s=1, c="lightgray")
        ax.scatter(*X[~noise].T, s=2, c=labels[~noise] % 10, cmap="tab10", vmin=0, vmax=9)
        ax.set_title(f"{name} ({len(np.unique(labels[~noise]))} clusters)")
        ax.view_init(elev=20, azim=-60)
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def score(labels):
    return f"ARI (hors bruit) = {adjusted_rand_score(y[core], labels[core]):.3f}, ARI (global) = {adjusted_rand_score(y, labels):.3f}"

## HGP et HDBSCAN

In [ ]:
start = time.perf_counter()
model = HGPClusterer(K=2, min_cluster_size=50, verbose=True).fit(X)
print(f"HGP (K=2) : {time.perf_counter() - start:.2f} s, {score(model.labels_)}")

hdbscan = HDBSCAN(min_cluster_size=50, min_samples=10, copy=True).fit_predict(X)
print(f"HDBSCAN (min_samples=10) : {score(hdbscan)}")

show({"Vérité terrain": y, "HGP, K=2": model.labels_, "HDBSCAN": hdbscan})

## Ordre des simplexes

`K=1` correspond au single linkage sur la triangulation de Delaunay. Les ordres supérieurs relient des faces par des triangles, puis des tétraèdres.

In [ ]:
for K in (1, 2, 3):
    start = time.perf_counter()
    labels = HGPClusterer(K=K, min_cluster_size=50).fit_predict(X)
    print(f"K={K} : {time.perf_counter() - start:.2f} s, {len(np.unique(labels[labels >= 0]))} clusters, {score(labels)}")

## Raffinement sans recalcul

`refine_clusters` sélectionne de nouveau les clusters sur la hiérarchie calculée par `fit`, sans refaire la triangulation. On peut ainsi prendre les feuilles de l'arbre condensé, couper la hiérarchie à différents rayons, ou découper dynamiquement selon une règle fournie par l'utilisateur. Ici, un cluster est découpé dès qu'au moins deux de ses enfants comptent 400 points ou plus.

In [ ]:
def large_children(parent_points, children_points):
    return sum(len(c) >= 400 for c in children_points) >= 2


for radius in (0.05, 0.08, 0.2):
    labels = model.refine_clusters(radius)
    print(f"coupe r={radius} : {len(np.unique(labels[labels >= 0]))} clusters, {score(labels)}")

panels = {}
for name, method, splitting in [
    ("feuilles", "leaf", None),
    ("coupe r=0.08", 0.08, None),
    ("EOM + découpage", "eom", large_children),
]:
    start = time.perf_counter()
    panels[name] = model.refine_clusters(method, splitting=splitting).copy()
    print(f"{name} : {time.perf_counter() - start:.3f} s, {score(panels[name])}")

show(panels)